In [ ]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

In [ ]:
%%bash
# Lever A — batch-size sweep at fixed n=6, UTD=4
# 验证 SAC update 是否 launch-bound:若 mean_ms 随 batch 增大近似不变,
# 则可直接通过增大 batch 拿到大幅 wallclock 收益(零代码改动)。
mkdir -p experiments/profile
for bs in 256 512 1024 2048; do
  python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 6 --vector-mode async --device cuda \
    --updates-per-step 4 --batch-size $bs \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n6_utd4_bs${bs}.json
done

In [ ]:
%%bash
# Lever B — n_envs sweep at fixed effective UTD ≈ 0.67
# effective UTD = updates_per_step / num_envs;保持与 thesis baseline (n=6, UTD=4)
# 等价的样本利用率,只看 wallclock 收益。L4 是 12 vCPU,推断 sweet spot 在 12-16。
# (n=6, UTD=4) baseline 已在 profile_train_env_completed.ipynb 跑过,这里不重复。

# n=12, UTD=8  → effective UTD = 8/12 ≈ 0.67
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 12 --vector-mode async --device cuda \
    --updates-per-step 8 --batch-size 256 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n12_utd8_bs256.json

# n=16, UTD=11 → effective UTD = 11/16 ≈ 0.69
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 16 --vector-mode async --device cuda \
    --updates-per-step 11 --batch-size 256 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n16_utd11_bs256.json

In [ ]:
%%bash
# Combo — Lever A + Lever B 同时打开
# n=12, UTD=8, batch=1024 → effective UTD ≈ 0.67,但每次 update 看到 4× 样本
# 这是 wallclock 最激进的免费午餐候选;若仍 ≤ 165 trans/s 说明假设不成立。
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 12 --vector-mode async --device cuda \
    --updates-per-step 8 --batch-size 1024 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n12_utd8_bs1024.json